In [13]:
# Import pandas for data loading and DataFrame operations
import pandas as pd
# Import os for checking file paths and file existence
import os

In [14]:
# Define the path of the original CSV dataset
file_path = "export.csv"
# Check whether the dataset file exists at the given path
print("File exists:", os.path.exists(file_path))

File exists: True


In [15]:
# Load only the first 10,000 rows of the large dataset
# to avoid loading the complete 14.5 GB file into memory
df_sample = pd.read_csv(
    file_path,
    nrows=10_000
)
# Confirm that the sample dataset was loaded successfully
print("Sample loaded successfully!")
# Display the number of rows in the sample
print("Rows:", len(df_sample))

# Display the number of columns in the sample
print("Columns:", len(df_sample.columns))

Sample loaded successfully!
Rows: 10000
Columns: 44


In [16]:
# Display all column names available in the sample dataset
print(df_sample.columns.tolist())

['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'Status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Council District', 'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Latitude', 'Longitude', 'Location']


In [17]:
# Select only the columns that are relevant to the
# Urban Service Resolution Intelligence Platform project
selected_columns = [
    "Created Date",
    "Closed Date",
    "Agency",
    "Agency Name",
    "Problem (formerly Complaint Type)",
    "Problem Detail (formerly Descriptor)",
    "Location Type",
    "Incident Zip",
    "City",
    "Borough",
    "Community Board",
    "Council District",
    "Police Precinct",
    "Open Data Channel Type",
    "Latitude",
    "Longitude",
    "Status",
    "Due Date"
]
# Create a new DataFrame containing only the selected project-relevant columns
df_project = df_sample[selected_columns].copy()
# Create a new DataFrame containing only the selected project-relevant columns
print("Selected rows:", len(df_project))

# Display the number of selected columns
print("Selected columns:", len(df_project.columns))

# Display the names of the selected columns
print("\nColumns being used:")
print(df_project.columns.tolist())

Selected rows: 10000
Selected columns: 18

Columns being used:
['Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Location Type', 'Incident Zip', 'City', 'Borough', 'Community Board', 'Council District', 'Police Precinct', 'Open Data Channel Type', 'Latitude', 'Longitude', 'Status', 'Due Date']


In [18]:
# Display the current working directory of the Jupyter Notebook
print("Current folder:")
print(os.getcwd())

# Display the complete absolute path of the CSV dataset
print("\nCSV full path:")
print(os.path.abspath("export.csv"))

Current folder:
C:\Users\kumar\Urban

CSV full path:
C:\Users\kumar\Urban\export.csv


In [19]:
#It is check of dataset structure
df_project.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Created Date                          10000 non-null  object 
 1   Closed Date                           3906 non-null   object 
 2   Agency                                10000 non-null  object 
 3   Agency Name                           10000 non-null  object 
 4   Problem (formerly Complaint Type)     10000 non-null  object 
 5   Problem Detail (formerly Descriptor)  9821 non-null   object 
 6   Location Type                         8925 non-null   object 
 7   Incident Zip                          9900 non-null   float64
 8   City                                  9318 non-null   object 
 9   Borough                               10000 non-null  object 
 10  Community Board                       10000 non-null  object 
 11  Council District

In [20]:
#How many data missing in each column
missing = df_project.isnull().sum() 

missing = missing[missing > 0].sort_values(ascending=False)

print("Missing values:")
print(missing)

Missing values:
Due Date                                9974
Closed Date                             6094
Location Type                           1075
City                                     682
Council District                         262
Latitude                                 211
Longitude                                211
Problem Detail (formerly Descriptor)     179
Incident Zip                             100
dtype: int64


In [21]:
# Convert date columns to datetime format
df_project["Created Date"] = pd.to_datetime(
    df_project["Created Date"],
    errors="coerce"
)

df_project["Closed Date"] = pd.to_datetime(
    df_project["Closed Date"],
    errors="coerce"
)

# Calculate resolution time in days
df_project["Resolution Days"] = (
    df_project["Closed Date"] - df_project["Created Date"]
).dt.total_seconds() / (24 * 60 * 60)

# Check the new target
print("Resolution Days created successfully!")
print(df_project["Resolution Days"].describe())

Resolution Days created successfully!
count    3906.000000
mean        0.076271
std         0.100834
min        -0.000266
25%         0.013273
50%         0.038449
75%         0.096508
max         0.704630
Name: Resolution Days, dtype: float64


In [22]:
# Check the date range and invalid resolution times

print("Created Date range:")
print(df_project["Created Date"].min(), "to", df_project["Created Date"].max())

print("\nClosed Date range:")
print(df_project["Closed Date"].min(), "to", df_project["Closed Date"].max())

print("\nMissing Resolution Days:", df_project["Resolution Days"].isna().sum())

print("Negative Resolution Days:",
      (df_project["Resolution Days"] < 0).sum())

print("\nNegative resolution examples:")
print(
    df_project.loc[
        df_project["Resolution Days"] < 0,
        ["Created Date", "Closed Date", "Resolution Days"]
    ].head()
)

Created Date range:
2026-08-20 07:38:40 to 2026-08-21 02:05:14

Closed Date range:
2026-08-20 07:41:35 to 2026-08-21 03:17:00

Missing Resolution Days: 6094
Negative Resolution Days: 5

Negative resolution examples:
            Created Date         Closed Date  Resolution Days
1997 2026-08-20 20:24:11 2026-08-20 20:24:00        -0.000127
3315 2026-08-20 18:35:23 2026-08-20 18:35:00        -0.000266
3573 2026-08-20 18:08:04 2026-08-20 18:08:00        -0.000046
5052 2026-08-20 15:15:01 2026-08-20 15:15:00        -0.000012
9329 2026-08-20 08:56:09 2026-08-20 08:56:00        -0.000104


In [23]:
# Remove invalid negative resolution times
df_project.loc[
    df_project["Resolution Days"] < 0,
    "Resolution Days"
] = None

print("Negative Resolution Days:",
      (df_project["Resolution Days"] < 0).sum())

print("Missing Resolution Days:",
      df_project["Resolution Days"].isna().sum())

Negative Resolution Days: 0
Missing Resolution Days: 6099


In [24]:
# Keep only rows where the target is available
df_model = df_project.dropna(
    subset=["Resolution Days"]
).copy()

print("Model rows:", len(df_model))
print("Model columns:", len(df_model.columns))

Model rows: 3901
Model columns: 19
